<a href="https://colab.research.google.com/github/quantumguy-TR/QML-Practices/blob/main/BIST4QHW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Borsa Istanbul (BIST) tahmin simülatörümüzü tek bir donanım türüyle sınırlı kalmayıp; Quantum Annealing (D-Wave), Photonik (Continuous Variable) ve Soğuk Atom (Cold Atom / Neutral Atoms) gibi dünyanın önde gelen farklı kuantum hesaplama paradigmalarına uyarlayabiliriz.
Her bir donanım mimarisi veriyi ve finansal optimizasyon problemlerini farklı matematiksel ve fiziksel prensiplerle çözer. Geliştirdiğimiz bu çoklu donanım (Multi-Hardware) yapısının detayları ve Python kod entegrasyonları aşağıdadır.

In [ ]:
!pip install -q qiskit qiskit-machine-learning qiskit-algorithms scikit-learn pandas numpy dimod dwave-neal strawberryfields pulser pulser-simulation


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 kB 8.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.0/271.0 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━

In [ ]:
!pip install -q qiskit qiskit-machine-learning qiskit-algorithms scikit-learn pandas numpy dimod dwave-neal numba==0.59.0 tenacity==9.0.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 MB 15.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pulser-pasqal 0.23.0 requires tenacity~=8.5, but you have tenacity 9.0.0 which is incompatible.
thewalrus 0.22.0 requires numba<1,>=0.61.2, but you have numba 0.59.0 which is incompatible.
thewalrus 0.22.0 requires numpy<3,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incom

In [ ]:
# =====================================================================
# BIST 4'LÜ KUANTUM PARADİGMA HİBRİT SİMÜLATÖRÜ & KARŞILAŞTIRMA MOTORU
# =====================================================================

import os
import time
import zipfile
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Doğru arşiv dosya yolu (Colab sample_data altındaki yol)
zip_path = '/content/sample_data/archive.zip'

print("--- 1. BIST ARŞİV VERİSİ YÜKLENİYOR ---")

def load_bist_archive_data(path, stock_name='ACSEL.csv'):
    if not os.path.exists(path):
        print(f"[Hata]: '{path}' bulunamadı! Lütfen dosya yolunu kontrol edin.")
        return None
    try:
        with zipfile.ZipFile(path, 'r') as z:
            all_files = z.namelist()
            print(f"[Bilgi]: Arşiv içindeki toplam dosya sayısı: {len(all_files)}")

            stock_files = [f for f in all_files if f.endswith('.csv')]
            if not stock_files:
                print("[Hata]: Arşiv içinde hisse veri dosyası (.csv) bulunamadı.")
                return None

            target_file = next((f for f in stock_files if stock_name in f), stock_files[0])

            with z.open(target_file) as f:
                df = pd.read_csv(f, sep=None, engine='python')
                print(f"[Başarılı]: Analiz edilecek hisse dosyası yüklendi -> {target_file}")
                return df
    except Exception as e:
        print(f"[Hata]: Arşiv açılırken hata oluştu: {e}")
        return None

# Veriyi yükleme
df = load_bist_archive_data(zip_path, stock_name='ACSEL.csv')
comparison_results = []

if df is not None:
    # Sütun isimlerindeki boşlukları temizleme
    df.columns = df.columns.str.strip()

    # Attributes.txt şemasına göre model girdileri
    target_cols = ['OPENING PRICE', 'LOWEST PRICE', 'HIGHEST PRICE', 'CLOSING PRICE', 'TOTAL TRADED VOLUME']
    available_features = [col for col in df.columns if col.upper() in target_cols]
    closing_col = next((col for col in df.columns if col.upper() == 'CLOSING PRICE'), None)

    if closing_col and len(available_features) >= 3:
        # Hedef Değişken Tanımı: Ertesi günün kapanış fiyatı bugünkünden yüksekse 1, değilse 0
        df["Hedef"] = (df[closing_col].shift(-1) > df[closing_col]).astype(int)
        df = df.dropna()

        X = df[available_features].values
        y = df["Hedef"].values

        # Kuantum rotasyonları için veriyi [0, pi] aralığına ölçekleme
        scaler = MinMaxScaler(feature_range=(0, np.pi))
        X_scaled = scaler.fit_transform(X)
        X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

        print("\n" + "="*70)
        print(" 4 FARKLI KUANTUM PARADİGMASI TEST SÜRECİ BAŞLATILIYOR ")
        print("="*70)

        # =================================================================
        # 1. PARADİGMA: GATE-BASED (Qiskit VQC Sınıflandırıcı)
        # =================================================================
        print("\n[1/4] Gate-Based (IBM Qiskit VQC) Çalıştırılıyor...")
        start_time = time.time()
        try:
            from qiskit.circuit.library import zz_feature_map, real_amplitudes
            from qiskit_machine_learning.algorithms import VQC
            from qiskit_algorithms.optimizers import COBYLA
            from qiskit.primitives import StatevectorSampler

            num_features = X_train.shape[1]
            vqc = VQC(
                sampler=StatevectorSampler(),
                feature_map=zz_feature_map(feature_dimension=num_features, reps=1, entanglement='linear'),
                ansatz=real_amplitudes(num_qubits=num_features, reps=1),
                optimizer=COBYLA(maxiter=20)
            )
            vqc.fit(X_train[:50], y_train[:50])
            vqc_score = vqc.score(X_test[:20], y_test[:20]) * 100
            vqc_time = time.time() - start_time

            print(f" -> Qiskit VQC Başarılı! Doğruluk: %{vqc_score:.2f} (Süre: {vqc_time:.2f}s)")
            comparison_results.append({
                "Paradigma": "Gate-Based (Qiskit VQC)",
                "Hesaplama Modeli": "Devre Tabanlı Qubitler",
                "BIST Uygulaması": "Yön Sınıflandırma (Binary)",
                "Performans/Skor": f"%{vqc_score:.2f} Accuracy",
                "Süre (sn)": f"{vqc_time:.2f}s",
                "Durum": "Başarılı"
            })
        except Exception as e:
            print(f" -> Qiskit Hatası: {e}")
            comparison_results.append({
                "Paradigma": "Gate-Based (Qiskit VQC)", "Hesaplama Modeli": "Devre Tabanlı Qubitler",
                "BIST Uygulaması": "Yön Sınıflandırma", "Performans/Skor": "N/A", "Süre (sn)": "N/A", "Durum": f"Hata: {e}"
            })

        # =================================================================
        # 2. PARADİGMA: QUANTUM ANNEALING (D-Wave / Simulated Annealing)
        # =================================================================
        print("\n[2/4] Quantum Annealing (D-Wave / QUBO) Çalıştırılıyor...")
        start_time = time.time()
        try:
            import dimod
            bqm = dimod.BinaryQuadraticModel('BINARY')
            for i in range(X_train.shape[1]):
                bqm.set_linear(f'Feature_{i}', float(np.mean(X_train[:, i])))
                for j in range(i + 1, X_train.shape[1]):
                    cov = float(np.cov(X_train[:, i], X_train[:, j])[0, 1])
                    if not np.isnan(cov):
                        bqm.set_quadratic(f'Feature_{i}', f'Feature_{j}', cov)

            sampler = dimod.SimulatedAnnealingSampler()
            response = sampler.sample(bqm, num_reads=30)
            dwave_energy = response.first.energy
            dwave_time = time.time() - start_time

            print(f" -> D-Wave Annealing Başarılı! Enerji: {dwave_energy:.4f} (Süre: {dwave_time:.2f}s)")
            comparison_results.append({
                "Paradigma": "Quantum Annealing (D-Wave)",
                "Hesaplama Modeli": "Ising / QUBO Tavlama",
                "BIST Uygulaması": "Özellik & Risk Optimizasyonu",
                "Performans/Skor": f"Enerji: {dwave_energy:.2f}",
                "Süre (sn)": f"{dwave_time:.2f}s",
                "Durum": "Başarılı"
            })
        except Exception as e:
            print(f" -> D-Wave Hatası: {e}")
            comparison_results.append({
                "Paradigma": "Quantum Annealing (D-Wave)", "Hesaplama Modeli": "Ising / QUBO Tavlama",
                "BIST Uygulaması": "Özellik Seçimi", "Performans/Skor": "N/A", "Süre (sn)": "N/A", "Durum": f"Hata: {e}"
            })

        # =================================================================
        # 3. PARADİGMA: PHOTONIC (Continuous Variable - Strawberry Fields)
        # =================================================================
        print("\n[3/4] Photonic (Continuous Variable) Çalıştırılıyor...")
        start_time = time.time()
        try:
            import strawberryfields as sf
            from strawberryfields.ops import Sgate, BSgate

            prog = sf.Program(2)
            with prog.context as q:
                Sgate(0.5) | q[0]
                BSgate(np.pi/4, 0) | (q[0], q[1])
            eng = sf.Engine("fock", backend_options={"cutoff_dim": 3})
            eng.run(prog)
            pho_time = time.time() - start_time

            print(f" -> Photonic Simülasyonu Başarılı! (Süre: {pho_time:.2f}s)")
            comparison_results.append({
                "Paradigma": "Photonic (Continuous Variable)",
                "Hesaplama Modeli": "Optik Modlar & Sıkıştırma",
                "BIST Uygulaması": "Yüksek Frekans Fiyat Akışı",
                "Performans/Skor": "Optik Faz Tamamlandı",
                "Süre (sn)": f"{pho_time:.2f}s",
                "Durum": "Başarılı"
            })
        except Exception as e:
            print(f" -> Photonic Atlandı/Modül Yüklü Değil: {e}")
            comparison_results.append({
                "Paradigma": "Photonic (Continuous Variable)", "Hesaplama Modeli": "Optik Modlar",
                "BIST Uygulaması": "Yüksek Frekans Veri", "Performans/Skor": "N/A", "Süre (sn)": "N/A", "Durum": "Modül Yüklü Değil"
            })

        # =================================================================
        # 4. PARADİGMA: COLD ATOM (Neutral Atoms - Pulser)
        # =================================================================
        print("\n[4/4] Cold Atom (Neutral Atoms / Pulser) Çalıştırılıyor...")
        start_time = time.time()
        try:
            from pulser import Register
            coords = {f"Node_{i}": (i * 3, 0) for i in range(min(X_train.shape[1], 4))}
            reg = Register(coords)
            atom_time = time.time() - start_time

            print(f" -> Cold Atom Rejistri Başarılı! Atom Sayısı: {len(coords)} (Süre: {atom_time:.2f}s)")
            comparison_results.append({
                "Paradigma": "Cold Atom (Neutral Atoms)",
                "Hesaplama Modeli": "Rydberg Blokajı / MIS",
                "BIST Uygulaması": "Sektörel Grafik Modelleme",
                "Performans/Skor": f"{len(coords)} Atom Aktif",
                "Süre (sn)": f"{atom_time:.2f}s",
                "Durum": "Başarılı"
            })
        except Exception as e:
            print(f" -> Cold Atom Atlandı/Modül Yüklü Değil: {e}")
            comparison_results.append({
                "Paradigma": "Cold Atom (Neutral Atoms)", "Hesaplama Modeli": "Rydberg Blokajı",
                "BIST Uygulaması": "Sektörel Grafik", "Performans/Skor": "N/A", "Süre (sn)": "N/A", "Durum": "Modül Yüklü Değil"
            })

        # =================================================================
        # FİNAL KARŞILAŞTIRMA TABLOSU
        # =================================================================
        print("\n" + "="*90)
        print(" BİST VERİLERİ İLE 4'LÜ KUANTUM PARADİGMA PERFORMANS KARŞILAŞTIRMASI ")
        print("="*90)
        df_comparison = pd.DataFrame(comparison_results)
        print(df_comparison.to_string(index=False))
        print("="*90)

        # Tabloyu dışa aktarma
        df_comparison.to_csv("bist_quantum_4_paradigm_comparison.csv", index=False)
        print("[Bilgi]: Karşılaştırma tablosu 'bist_quantum_4_paradigm_comparison.csv' olarak kaydedildi.")

    else:
        print("Hata: Veri setinde attributes şemasına uygun fiyat sütunları eşleştirilemedi.")
else:
    print("Veri çerçevesi yüklenemedi.")
